# Substation Shifting Proportionality Test

**Question:** Is the amount of load shifting per substation (Tempo − Control, TOU − Control) proportional to the number of households in that substation?

**Tests used:**
- **Pearson correlation** — tests linear association between n_households and total shifting
- **OLS regression with intercept** — verifies whether intercept ≈ 0 (necessary condition for proportionality)
- **OLS regression through origin** — directly models `shifting = k × n_households`; `k` is the per-household shift rate

**Why not ANOVA?** ANOVA compares group means across categories. Substation is not a treatment group — it's a continuous predictor. ANOVA can't express a proportional relationship.

**Why not chi-square?** Chi-square requires categorical data. Both our variables are continuous (kWh, household counts). Binning them would be arbitrary and lossy.

n = 21 substations (one observation per substation)

In [ ]:
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

BASE_DIR    = '/home/user/capstone_visuals'
ZIP_PATH    = os.path.join(BASE_DIR, 'dist_net.zip')
CONTROL_CSV = os.path.join(BASE_DIR, 'data', 'control_profile.csv')
TEMPO_CSV   = os.path.join(BASE_DIR, 'data', 'tempo_shifted_profile.csv')
TOU_ZIP     = os.path.join(BASE_DIR, 'data', 'tou_shifted_profile.zip')
EXTRACT_DIR = '/tmp/dist_net'
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')

HOUR_COLS = [str(i) for i in range(1, 25)]

def normalize_hid(x):
    try:    return str(int(float(x)))
    except: return None

print('Imports OK')

## 1. Build HID → Substation Mapping from Shapefiles

In [ ]:
# Extract network zip if needed
flag = os.path.join(EXTRACT_DIR, '.done')
if not os.path.exists(flag):
    print('Extracting dist_net.zip...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    open(flag, 'w').close()

# Load all H-nodes across substations to build HID → substation map
content = os.path.join(EXTRACT_DIR, 'content', 'output')
all_nodes = []

for folder in sorted(os.listdir(content)):
    fp = os.path.join(content, folder, f'{folder}-nodelist-HID.shp')
    if not os.path.exists(fp):
        continue
    gdf = gpd.read_file(fp)
    gdf['substation'] = folder
    all_nodes.append(gdf)

nodes = pd.concat(all_nodes, ignore_index=True)
h_nodes = nodes[nodes['label'] == 'H'].copy()
h_nodes['hid_key'] = h_nodes['hid'].apply(normalize_hid)
h_nodes = h_nodes.dropna(subset=['hid_key'])

# Household count per substation
hh_counts = h_nodes.groupby('substation')['hid_key'].nunique().rename('n_households')

print(f'Total H-nodes: {len(h_nodes):,}')
print(f'Substations:   {hh_counts.shape[0]}')
print()
print(hh_counts.sort_values(ascending=False).to_string())

## 2. Load CSVs and Compute Per-Substation Total Shifting

In [ ]:
# Load control
print('Loading control...')
ctrl = pd.read_csv(CONTROL_CSV)
ctrl['hid_key'] = ctrl['hid'].apply(normalize_hid)

# Load tempo
print('Loading tempo...')
tempo = pd.read_csv(TEMPO_CSV)
tempo['hid_key'] = tempo['hid'].apply(normalize_hid)

# Load TOU (from zip)
print('Loading TOU...')
with zipfile.ZipFile(TOU_ZIP) as z:
    with z.open('tou_shifted_profile.csv') as f:
        tou = pd.read_csv(f)
tou['hid_key'] = tou['hid'].apply(normalize_hid)

print(f'Control rows:  {len(ctrl):,}   unique HIDs: {ctrl["hid_key"].nunique():,}')
print(f'Tempo rows:    {len(tempo):,}   unique HIDs: {tempo["hid_key"].nunique():,}')
print(f'TOU rows:      {len(tou):,}   unique HIDs: {tou["hid_key"].nunique():,}')

In [ ]:
# Merge HID → substation into each dataset
hid_sub = h_nodes[['hid_key', 'substation']].drop_duplicates(subset='hid_key')

def add_substation(df):
    return df.merge(hid_sub, on='hid_key', how='inner')

ctrl_s  = add_substation(ctrl)
tempo_s = add_substation(tempo)
tou_s   = add_substation(tou)

print(f'After merge — Control: {len(ctrl_s):,} rows | Tempo: {len(tempo_s):,} rows | TOU: {len(tou_s):,} rows')

In [ ]:
# Compute total hourly load per (substation, date, hid) then aggregate
# 'shifting' = sum of (scenario - control) across all HID-days, per substation
# We sum all 24 hourly columns to get a daily total per HID, then sum over the substation

def substation_total_load(df):
    """Sum all hourly load across all HID-days, grouped by substation."""
    df = df.copy()
    # Sum hours 1-24 to get per-row (HID×day) total
    df['day_total'] = df[HOUR_COLS].sum(axis=1)
    return df.groupby('substation')['day_total'].sum()

load_ctrl  = substation_total_load(ctrl_s)
load_tempo = substation_total_load(tempo_s)
load_tou   = substation_total_load(tou_s)

# Build analysis dataframe — only substations present in all three datasets
df = pd.DataFrame({
    'n_households':   hh_counts,
    'load_ctrl':      load_ctrl,
    'load_tempo':     load_tempo,
    'load_tou':       load_tou,
}).dropna()

# Shifting = scenario minus control (negative = load reduced, positive = load added)
df['shift_tempo'] = df['load_tempo'] - df['load_ctrl']
df['shift_tou']   = df['load_tou']   - df['load_ctrl']

# Absolute shifting (magnitude regardless of direction)
df['abs_shift_tempo'] = df['shift_tempo'].abs()
df['abs_shift_tou']   = df['shift_tou'].abs()

print(f'Substations in analysis: {len(df)}')
print()
print(df[['n_households', 'shift_tempo', 'shift_tou', 'abs_shift_tempo', 'abs_shift_tou']]
      .sort_values('n_households', ascending=False)
      .to_string())

## 3. Pearson Correlation

Tests H₀: ρ = 0 (no linear association between n_households and shifting).  
We test both signed shifting and absolute shifting.

In [ ]:
def pearson_report(x, y, label_x, label_y):
    r, p = stats.pearsonr(x, y)
    n = len(x)
    # 95% CI via Fisher z-transform
    z  = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    ci_lo, ci_hi = np.tanh(z - 1.96*se), np.tanh(z + 1.96*se)
    print(f'  {label_y} ~ {label_x}')
    print(f'    n = {n}')
    print(f'    r = {r:+.4f}   (95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}])')
    print(f'    p = {p:.4f}  {"***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "(ns)"}')
    print()
    return r, p

print('=== Pearson Correlation: n_households vs shifting ===')
print()
print('--- TEMPO ---')
pearson_report(df['n_households'], df['shift_tempo'],     'n_households', 'shift_tempo (signed)')
pearson_report(df['n_households'], df['abs_shift_tempo'], 'n_households', 'shift_tempo (absolute)')

print('--- TOU ---')
pearson_report(df['n_households'], df['shift_tou'],     'n_households', 'shift_tou (signed)')
pearson_report(df['n_households'], df['abs_shift_tou'], 'n_households', 'shift_tou (absolute)')

## 4. OLS Regression

**With intercept:** checks whether the intercept is statistically distinguishable from zero.  
**Through origin (no intercept):** the strict proportionality model `shifting = k × n_households`.  

Uses `scipy.stats.linregress` (with intercept) and a manual OLS formula for the origin-constrained version.

In [ ]:
def ols_report(x, y, label_y, scenario):
    x_arr = x.values.astype(float)
    y_arr = y.values.astype(float)
    n = len(x_arr)

    # --- With intercept (scipy linregress) ---
    res = stats.linregress(x_arr, y_arr)
    print(f'  [{scenario}] {label_y} ~ n_households')
    print(f'  OLS WITH intercept:')
    print(f'    slope     = {res.slope:+.6f}  (se={res.stderr:.6f},  p={res.pvalue:.4f} {"***" if res.pvalue<0.001 else "**" if res.pvalue<0.01 else "*" if res.pvalue<0.05 else "(ns)"})')
    print(f'    intercept = {res.intercept:+.2f}  (se={res.intercept_stderr:.2f},  t={res.intercept/res.intercept_stderr:+.3f})')
    print(f'    R²        = {res.rvalue**2:.4f}')

    # --- Through origin: k = (x·y) / (x·x) ---
    k = np.dot(x_arr, y_arr) / np.dot(x_arr, x_arr)
    y_hat = k * x_arr
    residuals = y_arr - y_hat
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_arr - y_arr.mean())**2)
    r2_origin = 1 - ss_res / ss_tot          # note: can be negative for origin model
    # SE of k (no-intercept OLS)
    sigma2 = ss_res / (n - 1)                 # 1 free parameter
    se_k   = np.sqrt(sigma2 / np.dot(x_arr, x_arr))
    t_k    = k / se_k
    p_k    = 2 * stats.t.sf(abs(t_k), df=n-1)

    print(f'  OLS THROUGH ORIGIN (strict proportionality):')
    print(f'    k (kWh/household) = {k:+.6f}  (se={se_k:.6f},  t={t_k:+.3f},  p={p_k:.4f} {"***" if p_k<0.001 else "**" if p_k<0.01 else "*" if p_k<0.05 else "(ns)"})')
    print(f'    R²                = {r2_origin:.4f}')
    print()
    return res, k, se_k

print('=== OLS Regression ===')
print()
print('--- SIGNED SHIFTING ---')
res_tempo,  k_tempo,  se_tempo  = ols_report(df['n_households'], df['shift_tempo'],     'shift_tempo',     'TEMPO')
res_tou,    k_tou,    se_tou    = ols_report(df['n_households'], df['shift_tou'],       'shift_tou',       'TOU'  )

print('--- ABSOLUTE SHIFTING ---')
res_atempo, k_atempo, se_atempo = ols_report(df['n_households'], df['abs_shift_tempo'], 'abs_shift_tempo', 'TEMPO')
res_atou,   k_atou,   se_atou   = ols_report(df['n_households'], df['abs_shift_tou'],   'abs_shift_tou',   'TOU'  )

## 5. Visualisation

In [ ]:
BG   = '#06090f'
TXT  = '#c8d8e8'
DIM  = '#2a3f55'
C_TEMPO = '#ff8c00'
C_TOU   = '#00b4d8'
C_FIT   = '#ffffff'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'text.color': TXT, 'axes.labelcolor': TXT,
    'xtick.color': TXT, 'ytick.color': TXT,
    'axes.edgecolor': DIM, 'grid.color': DIM,
    'font.family': 'monospace', 'font.size': 9,
})

x_range = np.linspace(0, df['n_households'].max() * 1.05, 200)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), facecolor=BG)
fig.suptitle('Substation Shifting vs. Household Count\n'
             'Orange = Tempo   |   Cyan = TOU   |   Solid = with-intercept fit   |   Dashed = through-origin fit',
             color='white', fontsize=11, fontweight='bold', y=1.01)

configs = [
    (axes[0, 0], df['shift_tempo'],     C_TEMPO, 'TEMPO',  'Signed Shifting (kWh)',   res_tempo,  k_tempo ),
    (axes[0, 1], df['shift_tou'],       C_TOU,   'TOU',    'Signed Shifting (kWh)',   res_tou,    k_tou   ),
    (axes[1, 0], df['abs_shift_tempo'], C_TEMPO, 'TEMPO',  'Absolute Shifting (kWh)', res_atempo, k_atempo),
    (axes[1, 1], df['abs_shift_tou'],   C_TOU,   'TOU',    'Absolute Shifting (kWh)', res_atou,   k_atou  ),
]

for ax, y_data, color, scenario, ylabel, res_obj, k_val in configs:
    x_vals = df['n_households'].values
    y_vals = y_data.values

    r, p = stats.pearsonr(x_vals, y_vals)
    sig   = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    # Scatter
    ax.scatter(x_vals, y_vals, color=color, s=55, zorder=5, alpha=0.85,
               edgecolors='white', linewidths=0.4)

    # Label each point with substation ID
    for sub, xi, yi in zip(df.index, x_vals, y_vals):
        ax.annotate(sub, (xi, yi), textcoords='offset points', xytext=(5, 3),
                    fontsize=5.5, color=TXT, alpha=0.7)

    # With-intercept fit line
    y_fit  = res_obj.slope * x_range + res_obj.intercept
    ax.plot(x_range, y_fit, color=C_FIT, lw=1.3, alpha=0.7, zorder=4, label='OLS fit')

    # Through-origin fit line
    y_orig = k_val * x_range
    ax.plot(x_range, y_orig, color=C_FIT, lw=1.3, alpha=0.7, zorder=4,
            linestyle='--', label='Through origin')

    ax.axhline(0, color=DIM, lw=0.8, zorder=2)
    ax.set_xlabel('Households per substation', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(f'{scenario}  |  r = {r:+.3f}  (p = {p:.3f}, {sig})', fontsize=9)
    ax.grid(lw=0.35, alpha=0.5)
    ax.legend(fontsize=7, facecolor='#0c1622', edgecolor=DIM, labelcolor=TXT)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, '07_shifting_proportionality.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out_path}')

## 6. Summary Table

In [ ]:
rows = []
for scenario, y_col, res_obj, k_val, se_k_val in [
    ('Tempo', 'shift_tempo',     res_tempo,  k_tempo,  se_tempo ),
    ('TOU',   'shift_tou',       res_tou,    k_tou,    se_tou   ),
    ('Tempo', 'abs_shift_tempo', res_atempo, k_atempo, se_atempo),
    ('TOU',   'abs_shift_tou',   res_atou,   k_atou,   se_atou  ),
]:
    measure = 'Signed' if 'abs' not in y_col else 'Absolute'
    x = df['n_households'].values.astype(float)
    y = df[y_col].values.astype(float)
    r, p = stats.pearsonr(x, y)
    rows.append({
        'Scenario': scenario,
        'Shifting': measure,
        'Pearson r': round(r, 4),
        'p-value':  round(p, 4),
        'Significant (α=0.05)': 'Yes' if p < 0.05 else 'No',
        'OLS slope (with int.)': round(res_obj.slope, 4),
        'OLS intercept': round(res_obj.intercept, 2),
        'OLS R² (with int.)': round(res_obj.rvalue**2, 4),
        'k (origin, kWh/HH)':  round(k_val, 4),
        'k SE': round(se_k_val, 4),
    })

summary = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
print(summary.to_string(index=False))